# Implement Ring Attention for Long Contexts

**Difficulty**: 🟣 Expert

**Companies**: Anthropic, Google, Meta, xAI

---

### Problem Statement

For extremely long sequences (100K+ tokens), even FlashAttention hits memory limits because each device must hold the **full KV cache**. **Ring Attention** (Liu et al., 2023) solves this by distributing the sequence across devices in a **ring topology**.

The key idea:
- The sequence is split across `num_devices` devices. Device `i` holds Q chunk `i`, K chunk `i`, and V chunk `i`.
- Each device computes attention of its local Q against the **current** K,V chunk.
- Then K,V chunks are **passed to the next device** in the ring (device 0 -> 1 -> 2 -> ... -> 0).
- After `num_devices` ring steps, every Q chunk has attended to ALL K,V chunks.
- **Online softmax** is used to correctly combine partial attention results across ring steps.

This means each device only needs memory for:
- Its local Q chunk (permanent)
- One K,V chunk at a time (received from ring, temporary)
- Running softmax statistics (max, sum, output accumulator)

Peak memory per device: `O(N / num_devices)` instead of `O(N)`.

---

### Requirements

1. **Sequence Splitting** — Split Q, K, V across `num_devices` along the sequence dimension.
2. **Ring Step** — Implement `ring_step()` that computes attention of local Q against a received K,V chunk, updating online softmax statistics.
3. **Ring Communication** — Simulate the ring by rotating K,V chunks: device `i` sends to device `(i+1) % num_devices`.
4. **Causal Masking** — When a K,V chunk comes from a future sequence position relative to Q, it should be masked (skipped).
5. **Correctness** — Output must match standard full attention.

---

### Constraints

- ✅ Simulate multi-device with lists of tensors (no actual distributed communication)
- ✅ Must work on CPU
- ✅ Must handle causal and non-causal attention
- ❌ No device should ever hold more than `2/num_devices` of the full KV at any time (its own chunk + one received chunk)

---

<details>
  <summary>💡 Hint</summary>

  **Ring step with online softmax:**
  ```
  def ring_step(local_q, received_k, received_v, running_max, running_sum, running_output, scale):
      # Compute attention scores for this K,V chunk
      S = local_q @ received_k^T * scale     # (chunk_len, chunk_len)
      
      # Online softmax update (same as FlashAttention)
      block_max = S.max(dim=-1, keepdim=True)
      new_max = max(running_max, block_max)
      correction = exp(running_max - new_max)
      P = exp(S - new_max)
      
      running_sum = running_sum * correction + P.sum(dim=-1, keepdim=True)
      running_output = running_output * correction + P @ received_v
      running_max = new_max
      
      return running_max, running_sum, running_output
  ```
  
  **Ring rotation:** After each step, K,V chunks shift: `new_kv[i] = old_kv[(i-1) % n]`
  
  **Causal masking:** Device `i` with Q chunk `i` should skip K,V from chunk `j` if `j > i`
  (those are future tokens that Q should not attend to).

</details>

---

In [ ]:
import torch
import torch.nn.functional as F
import math
from typing import List, Tuple, Optional

In [ ]:
# Test data
torch.manual_seed(42)

# Dimensions
seq_len = 64        # total sequence length
head_dim = 32       # dimension per head
num_devices = 4     # number of simulated devices
chunk_len = seq_len // num_devices  # tokens per device

# Create Q, K, V: (seq_len, head_dim) — single head for clarity
Q = torch.randn(seq_len, head_dim)
K = torch.randn(seq_len, head_dim)
V = torch.randn(seq_len, head_dim)

print(f"Sequence length: {seq_len}")
print(f"Number of devices: {num_devices}")
print(f"Chunk length per device: {chunk_len}")
print(f"Head dimension: {head_dim}")
print(f"\nQ shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

In [ ]:
def standard_attention(Q, K, V, causal=False):
    """
    Standard full attention: O(N^2) memory.
    
    Args:
        Q, K, V: (seq_len, head_dim)
        causal: whether to apply causal masking
    Returns:
        output: (seq_len, head_dim)
    """
    d = Q.shape[-1]
    scores = Q @ K.T / math.sqrt(d)  # (N, N)
    
    if causal:
        N = Q.shape[0]
        mask = torch.triu(torch.ones(N, N, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float('-inf'))
    
    attn_weights = torch.softmax(scores, dim=-1)  # (N, N)
    output = attn_weights @ V  # (N, d)
    return output

In [ ]:
def ring_step(
    local_q: torch.Tensor,
    received_k: torch.Tensor,
    received_v: torch.Tensor,
    running_max: torch.Tensor,
    running_sum: torch.Tensor,
    running_output: torch.Tensor,
    scale: float,
    causal_mask: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    One step of ring attention: compute attention of local_q against received_k, received_v
    and update the online softmax accumulators.
    
    Args:
        local_q: (chunk_len, head_dim) — this device's Q chunk
        received_k: (chunk_len, head_dim) — K chunk received from ring
        received_v: (chunk_len, head_dim) — V chunk received from ring
        running_max: (chunk_len, 1) — current running max for online softmax
        running_sum: (chunk_len, 1) — current running sum for online softmax
        running_output: (chunk_len, head_dim) — accumulated output
        scale: 1/sqrt(d)
        causal_mask: optional (chunk_len, chunk_len) bool mask, True = masked
    Returns:
        updated (running_max, running_sum, running_output)
    """
    # TODO: Compute attention scores
    # S = local_q @ received_k^T * scale   ->  (chunk_len, chunk_len)
    ...
    
    # TODO: Apply causal mask if provided (set masked positions to -inf)
    ...
    
    # TODO: Online softmax update
    # 1. block_max = S.max(dim=-1, keepdim=True).values
    # 2. new_max = torch.maximum(running_max, block_max)
    # 3. correction = exp(running_max - new_max)
    # 4. P = exp(S - new_max)
    # 5. running_sum = running_sum * correction + P.sum(dim=-1, keepdim=True)
    # 6. running_output = running_output * correction + P @ received_v
    # 7. running_max = new_max
    ...
    
    return running_max, running_sum, running_output

In [ ]:
class RingAttention:
    """
    Ring Attention: distributes sequence across devices in a ring topology.
    Each device computes local attention, then passes K,V to the next device.
    """
    
    def __init__(self, num_devices: int):
        self.num_devices = num_devices
    
    def _split_sequence(self, tensor: torch.Tensor) -> List[torch.Tensor]:
        """
        Split a (seq_len, head_dim) tensor into num_devices chunks.
        """
        # TODO: Split tensor into equal chunks along dim=0
        ...
    
    def _rotate_ring(self, chunks: List[torch.Tensor]) -> List[torch.Tensor]:
        """
        Rotate chunks in the ring: device i sends to device (i+1) % n.
        After rotation, device i holds what was on device (i-1) % n.
        """
        # TODO: Rotate the list so that chunks[i] = old_chunks[(i-1) % n]
        # This simulates each device sending its K,V to the next device in the ring
        ...
    
    def forward(self, Q, K, V, causal=False):
        """
        Compute ring attention.
        
        Args:
            Q, K, V: (seq_len, head_dim)
            causal: whether to apply causal masking
        Returns:
            output: (seq_len, head_dim)
        """
        N, d = Q.shape
        scale = 1.0 / math.sqrt(d)
        chunk_len = N // self.num_devices
        
        # TODO: Split Q, K, V across devices
        ...
        
        # TODO: Initialize online softmax accumulators for each device
        # running_max: list of (chunk_len, 1) tensors, init to -inf
        # running_sum: list of (chunk_len, 1) tensors, init to 0
        # running_output: list of (chunk_len, d) tensors, init to 0
        ...
        
        # Track which original chunk index each device currently holds for K,V
        # Initially, device i holds K,V chunk i
        # TODO: kv_source = list(range(self.num_devices))
        ...
        
        # TODO: Ring loop - num_devices steps
        # For each step:
        #   For each device i:
        #     1. Determine if this K,V chunk should be skipped (causal: skip if kv_source[i] > i)
        #     2. Build causal mask if needed (for the partial case where kv_source[i] == i)
        #     3. Call ring_step to update accumulators
        #   After all devices process, rotate K,V chunks in the ring
        #   Update kv_source tracking
        ...
        
        # TODO: Final normalization and concatenation
        # For each device: output_chunk = running_output / running_sum
        # Concatenate all chunks: torch.cat(output_chunks, dim=0)
        ...

print("RingAttention class defined.")

In [ ]:
# Validation
print("=" * 60)
print("VALIDATING RING ATTENTION")
print("=" * 60)

ring_attn = RingAttention(num_devices=num_devices)

# Test 1: Non-causal attention
print("\n--- Test 1: Non-Causal Attention ---")
ref_output = standard_attention(Q, K, V, causal=False)
ring_output = ring_attn.forward(Q, K, V, causal=False)
max_err = (ring_output - ref_output).abs().max().item()
print(f"Max absolute error: {max_err:.2e}")
assert torch.allclose(ring_output, ref_output, atol=1e-5), "Non-causal ring attention FAILED!"
print("PASSED")

# Test 2: Causal attention
print("\n--- Test 2: Causal Attention ---")
ref_causal = standard_attention(Q, K, V, causal=True)
ring_causal = ring_attn.forward(Q, K, V, causal=True)
max_err_causal = (ring_causal - ref_causal).abs().max().item()
print(f"Max absolute error: {max_err_causal:.2e}")
assert torch.allclose(ring_causal, ref_causal, atol=1e-5), "Causal ring attention FAILED!"
print("PASSED")

# Test 3: Different number of devices
print("\n--- Test 3: Different Device Counts ---")
for n_dev in [2, 4, 8]:
    ring = RingAttention(num_devices=n_dev)
    out = ring.forward(Q, K, V, causal=False)
    err = (out - ref_output).abs().max().item()
    assert torch.allclose(out, ref_output, atol=1e-5), f"Failed for {n_dev} devices"
    print(f"  {n_dev} devices: max_err={err:.2e} PASSED")

# Test 4: Memory analysis
print("\n--- Test 4: Memory Analysis ---")
for n_dev in [1, 2, 4, 8]:
    full_kv_mem = seq_len * head_dim * 2  # K + V, full
    per_device_kv = (seq_len // n_dev) * head_dim * 2  # local chunk
    ring_kv = per_device_kv * 2  # own chunk + one received chunk
    print(f"  {n_dev} devices: per-device KV = {ring_kv} elements "
          f"({ring_kv / full_kv_mem * 100:.0f}% of full, "
          f"{full_kv_mem / ring_kv:.1f}x reduction)")

print("\nAll tests passed!")